In [ ]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

os.chdir(r'A:\Coding\PycharmProjects\cryptoguard')

#load
nazario = pd.read_csv('data/general/Nazario_5.csv')
trec = pd.read_csv('data/general/email_text.csv')

#standardise
#Nazario: combine subject + body into single text field
nazario['text'] = (
    nazario['subject'].fillna('') + ' ' + nazario['body'].fillna('')
).str.strip()
nazario_clean = nazario[['text', 'label']].copy()

#TREC07: already has 'text' and 'label'
trec_clean = trec[['text', 'label']].copy()

#text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ''
    #remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    #remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    #remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)
    #remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    #lowercase
    text = text.lower()
    return text

nazario_clean['text'] = nazario_clean['text'].apply(clean_text)
trec_clean['text'] = trec_clean['text'].apply(clean_text)

#remove empty texts after cleaning
nazario_clean = nazario_clean[nazario_clean['text'].str.len() > 10]
trec_clean = trec_clean[trec_clean['text'].str.len() > 10]

#balance and sample TREC07
#use all of Nazario (~3k), sample TREC07 to get to ~16k total balanced

#how many phishing and legitimate do we have from Nazario?
naz_phish = nazario_clean[nazario_clean['label'] == 1]
naz_legit = nazario_clean[nazario_clean['label'] == 0]

print(f"Nazario — phishing: {len(naz_phish)}, legitimate: {len(naz_legit)}")

#target: 8000 of each class total
target_per_class = 8000
trec_phish_needed = target_per_class - len(naz_phish)
trec_legit_needed = target_per_class - len(naz_legit)

trec_phish = trec_clean[trec_clean['label'] == 1].sample(
    n=trec_phish_needed, random_state=42)
trec_legit = trec_clean[trec_clean['label'] == 0].sample(
    n=trec_legit_needed, random_state=42)

print(f"Sampling from TREC07 — phishing: {trec_phish_needed}, legitimate: {trec_legit_needed}")

#combine
df = pd.concat([naz_phish, naz_legit, trec_phish, trec_legit], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f"\nFinal dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nSample texts:")
print(df[['text', 'label']].head(5))

#truncate to 512 tokens (approximate — exact truncation handled by tokeniser)
df['text'] = df['text'].str[:2000]  # rough character limit before tokenisation

#train / validation / test split (80/10/10)
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Train label dist:\n{train_df['label'].value_counts()}")

#save splits
os.makedirs('data/processed', exist_ok=True)
train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)
df.to_csv('data/processed/full_dataset.csv', index=False)

Nazario — phishing: 1564, legitimate: 1500
Sampling from TREC07 — phishing: 6436, legitimate: 6500

Final dataset shape: (16000, 2)
Label distribution:
label
1    8000
0    8000
Name: count, dtype: int64

Sample texts:
                                                text  label
0  the best all new dating site on the net check ...      1
1  you registered to receive this and similar off...      1
2  to see the accounting records echo first line ...      1
3  14 unread inbox server message dear you have r...      1
4  earn with ebay come to work with ebay online h...      1

Train: 12800, Val: 1600, Test: 1600
Train label dist:
label
0    6400
1    6400
Name: count, dtype: int64

Saved to data/processed/
